# 01 · Gate notebook: is the experiment fair, and did the message work?

**Why this notebook exists:** before building anything, we prove two things on your own laptop:
1. **K1 (fairness):** the message was given at random, so the two groups are comparable.
2. **K2 (effect):** customers who got the message bought more than those who didn't.

**How to use:** click a cell, press **Shift+Enter** to run it, read the output, then move to the next. Save with **Ctrl+S** when done so Claude can review.

## Cell 1 · Load the data
**What it does:** reads two files and joins them into one table (one row per customer).

- `uplift_train` = who got the message (`treatment_flg` 1/0) and who bought afterwards (`target` 1/0)
- `clients` = customer details (age, gender, when they got their loyalty card)

**Expect:** 200,039 rows; treated share about 0.50.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

RAW = Path('..') / 'data_x5' / 'raw'          # folder with the downloaded files

exp = pd.read_csv(RAW / 'uplift_train.csv.gz')  # the experiment: message yes/no, bought yes/no
cli = pd.read_csv(RAW / 'clients.csv.gz',
                  parse_dates=['first_issue_date', 'first_redeem_date'])  # customer details

df = exp.merge(cli, on='client_id', how='left')  # attach details to each experiment customer

print('rows:', len(df), '| treated share:', round(df.treatment_flg.mean(), 4))
df.head()

## Cell 2 · K1: was the message given at random?
**Idea:** if a coin flip decided, both groups should look the same on everything we can measure.

**Standardised difference** = gap between group averages, divided by how spread out the values are. Under **0.05** means the groups are practically identical.

**Expect:** every number under 0.05 → K1 passes.

In [ ]:
df['age_clean'] = df.age.where(df.age.between(14, 100))                        # remove impossible ages
df['tenure_days'] = (pd.Timestamp('2019-03-01') - df.first_issue_date).dt.days  # how long a loyalty member
df['ever_redeemed'] = df.first_redeem_date.notna().astype(int)                 # ever used points? 1/0

for col in ['age_clean', 'tenure_days', 'ever_redeemed']:
    t = df.loc[df.treatment_flg == 1, col]   # got message
    c = df.loc[df.treatment_flg == 0, col]   # did not
    std_diff = (t.mean() - c.mean()) / np.sqrt((t.var() + c.var()) / 2)
    print(f'{col:14s} treated {t.mean():8.2f} | control {c.mean():8.2f} | std diff {std_diff:+.3f}')

print()
print(pd.crosstab(df.gender, df.treatment_flg, normalize='columns').round(3))  # gender mix per group

## Cell 3 · K2: did the message work?
**Lift** = buy rate with message − buy rate without.

**95% range:** the band where the true lift almost certainly lies. If the band does not touch zero, the effect is real, not luck.

**Expect:** about +3.3 points; range about +2.9 to +3.8 → K2 passes.

In [ ]:
t = df[df.treatment_flg == 1].target   # bought? (message group)
c = df[df.treatment_flg == 0].target   # bought? (no-message group)

lift = t.mean() - c.mean()
se = np.sqrt(t.var() / len(t) + c.var() / len(c))    # uncertainty of the difference

print(f'buy rate: message {t.mean():.4f} | no message {c.mean():.4f}')
print(f'lift: {lift*100:+.2f} points | 95% range [{(lift-1.96*se)*100:+.2f}, {(lift+1.96*se)*100:+.2f}]')
print(f'relative lift: {100*lift/c.mean():+.1f}%')

## Cell 4 · First hint of targeting value: lift by age group
**Why:** if some groups respond much more than others, sending messages only to the responsive ones saves money. That is the whole project in one table.

**Expect:** lift grows with age (roughly +1 point for the young, +6 for 65+).

In [ ]:
df['age_band'] = pd.cut(df.age_clean, [14, 25, 35, 45, 55, 65, 100])
rates = df.groupby(['age_band', 'treatment_flg'], observed=True).target.mean().unstack()
rates.columns = ['buy rate, no message', 'buy rate, message']
rates['lift (points)'] = (rates['buy rate, message'] - rates['buy rate, no message']) * 100
rates.round(3)

## Your notes (write 2–3 lines in your own words)
- K1 result:
- K2 result:
- What the age table suggests: